# Python, for people who know Rust

> The twenty percent of the language you need, the four ways it will surprise you, and the array library that is the real subject of the day.

Read this chapter at `/learn/02-python-for-rust-programmers/`. Exported from `src/content/chapters/02-python-for-rust-programmers.mdx` — edit there, not here.


Here's a secret about this chapter: you don't really need to learn Python.

You need to learn **NumPy**, and just enough Python to carry it around. So that's
how the day is arranged — forty minutes on the language, and then the rest of it
on arrays, because arrays are what machine learning is actually written in. Every
model in this course, including the ones in PyTorch, is arrays underneath.

The shift that matters isn't syntax. It's that **you stop writing loops.**

In Rust you express computation as iteration over elements, and you do it well.
In NumPy you express it as operations on *whole arrays*, and the loop happens
somewhere you never see — in C, across eight cores, faster than you could have
written it. Letting go of the loop is the whole adjustment.

## The language, briefly

Three sentences of orientation: everything is an object, everything is a
reference, and nothing is checked until it runs. That last one is going to feel
like walking off a cliff for about a day. It passes.

In [ ]:
def train(examples, lr=0.01, *, verbose=False):
    """Docstrings are a real expression, not a comment."""
    total = 0.0
    for i, (x, y) in enumerate(examples):
        total += (x - y) ** 2
        if verbose:
            print(f"  {i}: running total {total:.2f}")
    return total / len(examples)

train([(1.0, 0.8), (2.0, 2.4)], verbose=True)

Six things just happened that are worth naming out loud.

Indentation *is* the block structure — no braces, and a wrong indent is a syntax
error rather than a style complaint. Arguments after the bare `*` are
**keyword-only**, so callers have to write `verbose=True`; use that liberally,
because a bare `True` at a call site tells the reader nothing.
f-strings interpolate any expression you like.
`enumerate` pairs items with their indices, and
tuple unpacking pulls each pair apart right there
in the loop header. `**` is exponentiation, not a dereference. And `/` on two
integers gives you a float — `7 / 2` is `3.5`, where `7 // 2` is `3`.

Four differences that will each cost you an hour if nobody says them out loud, so
here they are out loud.

**No ownership, no borrows, no `mut`.** Everything is a reference to a heap
object, and assignment rebinds a name rather than copying. So `b = a` followed by
`b.append(1)` mutates what `a` sees. There is no compiler to catch this. There is
no compiler at all.

**No `Option`, no `Result`.** Absence is `None`. Failure is an exception that
propagates upward until something catches it. There's no `?`, and nothing warns
you that you ignored a failure path.

**Annotations are not types.** `def f(x: int)` is a comment that tooling can
read. Passing a string works perfectly right up until something inside does
arithmetic on it.

**No overflow. Ever.** Python integers are arbitrary precision — `2 ** 1000` is
an exact number, and it will happily print all 302 digits. NumPy integers, on the
other hand, are fixed-width and *do* wrap silently. Mixing the two is a genuinely
nasty edge, and you will meet it eventually.

### Comprehensions replace your iterator chains

In [ ]:
xs = list(range(10))
squares = [x * x for x in xs if x % 2 == 0]
lookup  = {name: i for i, name in enumerate("abc")}
squares, lookup

A list comprehension is
`iter().filter().map().collect()` with the collect implied. Dict and set
comprehensions use the same syntax with `{}`. Swap the brackets for parentheses
and you get a generator — lazy, exactly like a Rust
iterator you haven't consumed yet.

The rule of thumb transfers too: one `for` and one `if` reads better than a loop.
Two of each reads worse than a loop. Write the loop.

### The pieces you'll meet constantly

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    lr: float = 1e-3
    epochs: int = 5

cfg = Config(lr=0.01)
print(cfg)
print({**cfg.__dict__, "epochs": 10})   # ** spreads a dict

`@dataclass` is a decorator — it
takes the class, and hands back a modified one with a constructor, a `repr` and
equality all generated from the annotations. It's `#[derive(Debug, PartialEq)]`
with a slightly different spelling.

`**` spreads a dict, and that's the mechanism behind every `**kwargs` you'll ever
see in a library signature — and there are a lot of them.

Two more you'll read every single day: `with`, which
is scoped setup and teardown, and `pathlib.Path`, where `/`
joins path components. Yes, really — they overloaded division to build paths, and
after a week you'll find you rather like it.

## NumPy, which is the actual subject

Right. Everything from here is the point of the day.

In [ ]:
import numpy as np

a = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print(a.shape, a.dtype, a.ndim, a.size)
a * 2 + 1

An `ndarray` is one contiguous, fixed-size, single-dtype
block of memory, plus a shape describing how to walk it. Think of a `Vec<f64>`
you can't push to, carrying a `shape: [usize; N]`, with every arithmetic operator
overloaded to apply elementwise.

Now, I could tell you it's faster. But you have a Python right there in the page,
so let's just find out.

In [ ]:
import time

n = 300_000
py_list = list(range(n))
np_arr  = np.arange(n)

t = time.perf_counter()
_ = [x * 2 for x in py_list]
py_ms = (time.perf_counter() - t) * 1000

t = time.perf_counter()
_ = np_arr * 2
np_ms = (time.perf_counter() - t) * 1000

print(f"python list : {py_ms:7.2f} ms")
print(f"numpy array : {np_ms:7.2f} ms   ({py_ms / np_ms:.0f}x faster)")

The gap isn't because Python is bad at multiplication. It's a *memory layout*
story, and it's a nice one.

A Python list of a million floats is a million **pointers**, aimed at a million
separately allocated, individually type-tagged objects scattered across the heap.
Every `x * 2` has to chase a pointer, ask the object what type it is, look up the
right multiply, allocate a new object for the answer, and store another pointer.

A `float64` array is eight megabytes of doubles, laid end to end, and one
pointer. The CPU can walk it in a straight line, pull 4 or 8 values into a
register at a time, and never ask a single question about types.

You already have the intuition for this — it's `Vec<Box<dyn Any>>` versus
`Vec<f64>`, and it's the reason you'd never write the first one. What's rather
delightful is that this is the *same* insight, arrived at independently, that
makes columnar databases fast, and game engines fast, and Polars fast. Put the
same kind of thing next to the same kind of thing, and hardware built in 1978
still rewards you for it.

NumPy is simply how Python buys its way back into that world.

### Shape is the whole game

I want to be direct about this: almost every bug you hit for the next fortnight
will be a shape bug. Not a logic bug. A shape bug. Learn to read shapes and
you'll fix in ten seconds what would otherwise eat an evening.

In [ ]:
a = np.arange(12)
print("flat      ", a.shape)
print("as 3x4    ", a.reshape(3, 4).shape)
print("as ?x2    ", a.reshape(-1, 2).shape)     # -1 means 'you work it out'
print("transposed", a.reshape(3, 4).T.shape)

`reshape` is free. The buffer doesn't move — only the
metadata describing how to walk it changes. That's why you'll see it thrown
around so casually.

The other half of shape is `axis`, and there's exactly one
thing to remember, so let me put it on its own line:

**`axis=k` is the dimension that disappears.**

In [ ]:
m = np.arange(6).reshape(2, 3)
print(m)
print("sum(axis=0) collapses the 2 ->", m.sum(axis=0), m.sum(axis=0).shape)
print("sum(axis=1) collapses the 3 ->", m.sum(axis=1), m.sum(axis=1).shape)

In practice `axis=0` means "down the rows, across the batch" — the average of one
feature over every example. And `axis=-1` means "along the last dimension," which
is where class scores live, so `preds.argmax(axis=-1)` is how a matrix of scores
becomes a vector of predicted labels. You'll write that line a hundred times.

### Broadcasting

This is the one piece of NumPy with no Rust analogue at all, and it's on
essentially every line of every neural network ever written.

In [ ]:
rows = np.arange(3).reshape(3, 1)   # shape (3, 1)
cols = np.arange(4)                 # shape (4,)
print(rows + cols)                  # -> (3, 4)

A `(3,1)` plus a `(4,)` gave a `(3,4)`. Nothing was copied to make that happen.

Broadcasting stretches mismatched dimensions of size
`1` up to whatever's needed, without allocating. The rule, applied right to left:
dimensions are compatible if they're equal, or if one of them is `1`.

Its most important use is completely undramatic — adding a bias vector to a batch:

In [ ]:
batch = np.ones((32, 4))     # 32 examples, 4 features
bias  = np.array([10., 20., 30., 40.])   # one per feature
(batch + bias).shape, (batch + bias)[0]

Thirty-two rows, the same four numbers added to each one. As a loop that's five
lines. As broadcasting it's `+`. And it is the line inside every linear layer
you will ever meet.

Broadcasting is also the most common silent bug in the entire field, because the
failure mode isn't an exception. It's a number.

In [ ]:
pred  = np.array([1., 2., 3.])            # shape (3,)
truth = np.array([[1.], [2.], [3.]])      # shape (3, 1)  <- a stray column
err = pred - truth
print("expected shape (3,), got", err.shape, "and", err.size, "numbers")
print(err)

Three predictions minus three truths gave you **nine numbers**. And `err.mean()`
is now a perfectly plausible-looking float that means absolutely nothing.

I'm not being dramatic — you will write this bug. Everyone does. The habit that
saves you: when a metric looks weird, print `.shape` before you print anything
else. Before you think, even.

### The operations that matter

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(5, 3))
w = rng.normal(size=3)

print("X @ w        ", (X @ w).shape, "  <- matrix multiply, the workhorse")
print("X.mean(0)    ", X.mean(axis=0).round(2))
print("X > 0        ", (X > 0).sum(), "positive entries")
print("X[X > 0][:3] ", X[X > 0][:3].round(2), " <- boolean indexing")

`@` is matrix multiplication; `*` is elementwise. Mixing
those up is the second most common bug, and mercifully — unlike broadcasting — it
usually does throw.

Boolean indexing selects wherever a mask is true,
and here's a small thing I find quietly beautiful: `(a == b).mean()` — the
fraction of positions that match — is the *entire* implementation of accuracy.
Comparison makes booleans, booleans are 1 and 0, and the mean of those is a
percentage. One expression, no branches.

`default_rng(0)` is how you make a run reproducible.
Please use it every single time. The alternative is being genuinely unable to
tell an improvement from luck, which is a bad place to spend a week.

### The one that will actually bite you

In [ ]:
a = np.arange(6)
view = a[2:5]        # a VIEW into a's memory
view[0] = 999
print("a is now", a)

b = np.arange(6)
copy = b[2:5].copy() # explicit copy
copy[0] = 999
print("b is still", b)

Slicing a Python list copies. Slicing a NumPy array does
**not** — you get a view over the same buffer, and writing through it mutates the
original.

This is a deliberate performance decision, and yes: it is precisely the aliasing
that Rust's borrow checker exists to make impossible. Nothing here will stop you.
Nothing will even mention it. When you need independence, say `.copy()`, and say
it on purpose.

**"How do I know if I have a view or a copy?"** Slicing gives a view. Fancy
indexing (`a[[0, 2, 4]]`) and boolean indexing (`a[a > 0]`) give copies.
`.reshape()` gives a view when it can. If it matters, `arr.base is not None`
tells you it's a view of something.

**"I keep getting `shapes not aligned`."** That's `@` telling you the inner
dimensions disagree. For `A @ B`, `A.shape[-1]` must equal `B.shape[-2]`. Print
both shapes and read them next to each other — the mismatch is usually a missing
`.T`.

**"Comprehensions still feel unnatural."** Read them from the `for` outward:
`[x * x for x in xs if x % 2 == 0]` is "for each x in xs, if it's even, give me
x squared." The output expression sits at the front for the same reason
`.map()` sits at the end of a Rust chain — you read the transformation last,
even though it's written first.

**"I don't believe broadcasting isn't copying."** Fair. It's implemented with
*strides*: a stretched dimension is given a stride of zero, so walking along it
reads the same memory over and over. There's no copy because there's nothing to
copy — the array simply lies about how to move.

## Ten minutes of pandas

You need just enough to load a table and look at it honestly.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "hours":  [1.0, 2.0, 3.5, 4.0, 5.5, 6.0],
    "passed": [0, 0, 0, 1, 1, 1],
    "cohort": ["a", "a", "b", "b", "a", "b"],
})
df.head(3)

A `DataFrame` is a `Vec<Struct>` turned inside out: one
typed array per column, rather than one allocation per row. If you've used
Polars, this is the same design — Polars is written in Rust, and its API is a
tidier version of this one. You're allowed to think so out loud.

In [ ]:
print(df.dtypes.to_dict())
print()
print(df.groupby("cohort")["passed"].mean())

`groupby` before you model. Always. Here's the reasoning,
and it's worth more than it looks: if the target's average doesn't move across
the values of a column, that column carries no signal — and no model, however
clever, will invent signal that isn't there.

Ten minutes of `groupby` regularly saves an afternoon of training. It is the
cheapest thing in this entire course.

Two more: `df.loc` selects by label and `df.iloc` by
position, and `df["col"].values` hands you the underlying NumPy array — which is
what every model actually wants from you anyway.

You'll see these names constantly and it's nice not to be mystified by them.

**SciPy** sits on top of NumPy and adds the things a numerical analyst wants:
optimisation, statistics, sparse matrices, signal processing, linear algebra
beyond the basics. When you need a chi-squared test or a sparse matrix, it's
already installed.

**Polars** is a DataFrame library written in Rust, with a lazy query optimiser
and genuinely better ergonomics than pandas. If you're doing heavy tabular work
in a Rust-adjacent shop, it's a real option. This course uses pandas purely
because every tutorial, Stack Overflow answer and paper you'll meet uses pandas,
and being able to read them matters more here than being fast.

**JAX** is NumPy with three superpowers bolted on: automatic differentiation
(chapter 9's subject, done for you), just-in-time compilation to GPU/TPU, and
automatic vectorisation. It's beloved in research and rare in production. If you
enjoy chapter 9, you'll enjoy JAX.

**`einsum`** deserves a mention because it looks like line noise and isn't:

In [ ]:
import numpy as np
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)

print("normal :", (A @ B).shape)
print("einsum :", np.einsum("ij,jk->ik", A, B).shape)

Read `"ij,jk->ik"` as: first array is indexed by `i` and `j`, second by `j` and
`k`, and I want a result indexed by `i` and `k`. Any index that appears on the
left but not the right gets summed over. That's the whole rule.

It's verbose for a matrix multiply, and it's a lifesaver for the six-dimensional
tensor contractions inside a transformer, where you genuinely cannot remember
which axis is which. You'll see it in chapter 13's neighbourhood.

Do these before moving on. They're not busywork — they're precisely the
operations the next four chapters assume you can do without thinking.

In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(size=(100, 3))      # 100 examples, 3 features
y = rng.integers(0, 2, size=100)   # binary labels

# 1. Standardise every column: subtract its mean, divide by its std.
#    Do it with broadcasting, in one line, no loops.
Xs = ...

# 2. What fraction of labels are 1?  (one expression, no sum())

# 3. Compute the mean of each feature *for the rows where y == 1*.

# 4. Make a (100, 4) array by adding a column of ones to X.
#    Look up np.column_stack or np.hstack.

print("replace the ... above and re-run")

For 1, remember `X.mean(axis=0)` has shape `(3,)` and `X` has shape `(100, 3)` —
broadcasting will do the rest if you just write the obvious thing.

For 2, think about what the mean of a bunch of 0s and 1s actually *is*.

For 3, boolean indexing gives you the rows you want; then take a mean over
axis 0.

In [ ]:
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
print("1.", Xs.mean(axis=0).round(6), Xs.std(axis=0).round(6))

print("2.", y.mean())

print("3.", X[y == 1].mean(axis=0).round(3))

X1 = np.column_stack([np.ones(len(X)), X])
print("4.", X1.shape, X1[0].round(3))

Number 1 is the single most common preprocessing step in the entire field, and
broadcasting is doing all the work: `X` is `(100, 3)`, `X.mean(axis=0)` is
`(3,)`, and the subtraction quietly stretches those three numbers across all
hundred rows. Note the means come out as `-0.0` and the stds as exactly `1.0` —
that's the point of the operation.

Number 2 is the small delight from earlier: the mean of a 0/1 array *is* the
proportion. No counting required.

Number 4 is a trick you'll see in the very next chapter. Adding a column of ones
lets you fold the bias term into the weight vector, so that $wx + b$ collapses
into a single matrix multiply. It's a small piece of algebraic sleight of hand
and it makes the code noticeably cleaner.

Tomorrow, before we write another line of model: how to tell what kind of problem
you're looking at — and how to tell when you're not looking at one at all.